# Chapter 7 — Self-Attention: Learning Which Context Matters

> "The devil is in the details, and so is the attention."
> — Anonymous

---

> **Capsule summary.** Chapter 6 built a `FixedWindowModel` that concatenates the k most recent
> token embeddings and passes them through a single linear layer. Each position applies a fixed,
> learned transformation of its own token — independent of what occupies every other position.
> This chapter replaces those fixed positional transformations with a content-dependent mechanism:
> **self-attention**. Given a sequence of token embeddings, each position computes a query vector,
> compares it to every other position's key vector, and aggregates value vectors in proportion to
> those similarities. The result is a context representation that depends on the actual content of
> the sequence — not on fixed positional slots. We build a single-head self-attention module from
> scratch, wrap it in a language model, train it on tinyshakespeare, and compare its perplexity
> with the best fixed-window model from Chapter 6.

---

Chapter 6 ended with a precise limitation:

> Each position applies a fixed learned transformation of its own token embedding.
> Its contribution depends on the token at that position, but cannot adapt based
> on the identities of tokens at other positions.

The core equation was:

$$z_i = b_i + \sum_{j=1}^{k} W_{i,j} \cdot e(x_j)$$

Each positional block $W_{i,j}$ is fixed after training. The embedding $e(x_j)$ changes with
the token, but the transformation applied to it does not depend on $x_{j'}$ for any $j' \neq j$.

Self-attention solves exactly this. Instead of fixed blocks, it computes a **similarity score**
between every pair of positions using their content — then uses those scores to weight which
positions to aggregate. The transformation each position receives is computed fresh for every input.

Open [`ch07_attention.ipynb`](https://github.com/asbassan/AmarLMFP/blob/main/ch07_attention.ipynb)
and follow along.

---


## Setup — data, vocabulary, device


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import time

# ── load tinyshakespeare (same corpus as Chapter 6) ───────────────────────────
with open('tinyshakespeare.txt', 'r', encoding='utf-8') as f:
    text = f.read()

# ── character vocabulary ──────────────────────────────────────────────────────
chars     = sorted(set(text))
vocab_size = len(chars)
stoi      = {ch: i for i, ch in enumerate(chars)}
itos      = {i: ch for i, ch in enumerate(chars)}
encode    = lambda s: [stoi[c] for c in s]
decode    = lambda l: ''.join([itos[i] for i in l])

print(f"Corpus length : {len(text):,} characters")
print(f"Vocabulary    : {vocab_size} unique characters")

# ── train / validation split (90 / 10) ────────────────────────────────────────
data  = torch.tensor(encode(text), dtype=torch.long)
split = int(0.9 * len(data))
train_data = data[:split]
val_data   = data[split:]

# ── device ────────────────────────────────────────────────────────────────────
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device        : {device}")
print(f"Train tokens  : {len(train_data):,}")
print(f"Val tokens    : {len(val_data):,}")


---

## Part 1 — From Fixed Transformations to Content-Dependent Weighting


### Section 1 — What Chapter 6 cannot do


In [ ]:
# ── visualise the fixed-position limitation ──────────────────────────────────
#
# Chapter 6's FixedWindowModel forward pass:
#
#   emb = embed(x)            # (batch, k, embed_dim)
#   emb = emb.view(B, -1)     # (batch, k * embed_dim)  — flatten into fixed slots
#   out = head(emb)            # (batch, vocab_size)
#
# The contribution of position j to output token i:
#
#   score_i += W[i, j*D:(j+1)*D] · embedding(token at position j)
#
# W[i, j*D:(j+1)*D] is a fixed block — it does not change based on what
# token sits in any other position. Every forward pass uses the same block.
#
# DEMO: show that the same context position always gets the same weight block.

import torch, torch.nn as nn

class FixedWindowModel(nn.Module):
    'Minimal reproduction of the Chapter 6 architecture for comparison.'
    def __init__(self, vocab_size, k, embed_dim=64):
        super().__init__()
        self.k     = k
        self.embed = nn.Embedding(vocab_size, embed_dim)
        self.head  = nn.Linear(k * embed_dim, vocab_size)

    def forward(self, x):
        emb = self.embed(x)
        emb = emb.view(emb.size(0), -1)
        return self.head(emb)

# For k=4, the head weight matrix has shape (vocab_size, 4 * embed_dim).
# Block j = columns [j*64 : (j+1)*64] — always assigned to position j.
fw = FixedWindowModel(vocab_size, k=4)
W  = fw.head.weight.detach()
print(f"Head weight shape: {list(W.shape)}  = (vocab_size, k * embed_dim)")
print()
print("Each column block is PERMANENTLY assigned to one positional slot:")
for j in range(4):
    block_norm = W[:, j*64:(j+1)*64].norm(dim=1).mean().item()
    print(f"  Block {j}  cols {j*64:3d}–{(j+1)*64-1:3d}  (pos -{4-j})  avg norm={block_norm:.4f}")
print()
print("These blocks do not change based on what token occupies another slot.")
print("Self-attention replaces them with content-dependent scores.")


### Section 2 — The attention idea: score by similarity


In [ ]:
# ── the core idea of self-attention ──────────────────────────────────────────
#
# PROBLEM:  fixed positional blocks cannot route information based on content.
# SOLUTION: compute a similarity score between every pair of positions,
#           then use those scores to weight how much each position contributes.
#
# The key insight: if we project each token embedding into two spaces —
#   QUERY (q): "what am I looking for?"
#   KEY   (k): "what do I contain?"
# — then the dot product q_t · k_s measures how relevant position s is
# to position t. This score is computed FRESH for every input.
#
# A third projection produces the VALUE (v): "what do I contribute?"
# The output at position t is a weighted sum of all value vectors,
# where the weights are the softmax-normalised query-key similarity scores.
#
# Formally, for a sequence of T tokens with embedding dimension d_model:
#
#   Q = X W_Q        (T, d_k)  — one query vector per position
#   K = X W_K        (T, d_k)  — one key vector per position
#   V = X W_V        (T, d_v)  — one value vector per position
#
#   scores = Q K^T / sqrt(d_k)  (T, T)  — similarity of every pair
#   weights = softmax(scores)   (T, T)  — normalised attention weights
#   output = weights @ V        (T, d_v) — weighted value aggregation

# Simple toy example: 4-token sequence, d_model=8, d_k=d_v=4
torch.manual_seed(0)
T, d_model, d_k = 4, 8, 4

X   = torch.randn(T, d_model)        # (T, d_model) — token embeddings
W_Q = torch.randn(d_model, d_k)
W_K = torch.randn(d_model, d_k)
W_V = torch.randn(d_model, d_k)

Q = X @ W_Q    # (T, d_k)
K = X @ W_K    # (T, d_k)
V = X @ W_V    # (T, d_k)

scores  = Q @ K.T / math.sqrt(d_k)   # (T, T) — raw similarity scores
weights = F.softmax(scores, dim=-1)   # (T, T) — rows sum to 1
output  = weights @ V                 # (T, d_k) — context-aware representations

print(f"X shape      : {list(X.shape)}")
print(f"Q, K, V shape: {list(Q.shape)}")
print(f"scores shape : {list(scores.shape)}")
print(f"weights shape: {list(weights.shape)}")
print(f"output shape : {list(output.shape)}")
print()
print("Attention weight matrix (rows=queries, cols=keys):")
print("  Row t shows how much position t attends to each other position.")
for t in range(T):
    bar = '  '.join(f'{weights[t,s].item():.2f}' for s in range(T))
    print(f"  pos {t}: [{bar}]")
print()
print("Each row sums to 1.0 (softmax normalised).")
print("The weights depend on the content of Q and K — different input → different weights.")


---

## Part 2 — Building Single-Head Self-Attention


### Section 3 — Scaling by sqrt(d_k)


In [ ]:
# ── why we divide by sqrt(d_k) ────────────────────────────────────────────────
#
# The dot product q · k is a sum of d_k multiplied pairs.
# Its variance grows with d_k — larger d_k → larger raw scores.
# Large scores → softmax pushes one weight near 1 and all others near 0.
# This "peaky" distribution kills gradient flow: nearly all gradient goes
# to one position, and the others receive almost nothing.
#
# Dividing by sqrt(d_k) normalises the variance back to ~1 regardless of d_k,
# keeping the softmax in a well-behaved regime during early training.
# (Vaswani et al. 2017, Section 3.2.1)

torch.manual_seed(0)
d_k = 64

q = torch.randn(d_k)
k = torch.randn(d_k)

raw   = q @ k
scaled = raw / math.sqrt(d_k)

print(f"d_k = {d_k}")
print(f"Raw dot product   : {raw.item():.4f}")
print(f"Scaled dot product: {scaled.item():.4f}  (divided by sqrt({d_k})={math.sqrt(d_k):.1f})")
print()

# Show effect on softmax for a sequence of T=8 scores
T = 8
scores_raw    = torch.randn(T) * math.sqrt(d_k)   # large variance
scores_scaled = scores_raw / math.sqrt(d_k)        # normalised

w_raw    = F.softmax(scores_raw,    dim=-1)
w_scaled = F.softmax(scores_scaled, dim=-1)

print("Effect on softmax (8 positions):")
print(f"  Without scaling — max weight: {w_raw.max().item():.4f}  entropy: {(-w_raw * w_raw.log()).sum().item():.4f} bits")
print(f"  With scaling    — max weight: {w_scaled.max().item():.4f}  entropy: {(-w_scaled * w_scaled.log()).sum().item():.4f} bits")
print()
print("Scaled softmax distributes attention more evenly during early training.")


### Section 4 — Causal masking


In [ ]:
# ── causal (decoder) masking ──────────────────────────────────────────────────
#
# Language modelling is CAUSAL: at position t we must predict token t+1
# using only tokens 0..t. Position t must NOT be allowed to attend to
# positions t+1, t+2, ... — those tokens don't exist yet at inference time.
#
# Implementation: before softmax, set scores[t, s] = -inf for all s > t.
# After softmax, exp(-inf) = 0 — those positions receive zero weight.
#
# The mask is a lower-triangular boolean matrix: True where attention is ALLOWED.

T = 6
mask = torch.tril(torch.ones(T, T, dtype=torch.bool))   # (T, T)

print("Causal mask (True = attend, False = masked):")
for t in range(T):
    row = '  '.join('1' if mask[t, s].item() else '0' for s in range(T))
    print(f"  pos {t}: [{row}]  (can attend to positions 0..{t})")
print()

# Apply mask to scores
scores = torch.randn(T, T)
scores_masked = scores.masked_fill(~mask, float('-inf'))
weights = F.softmax(scores_masked, dim=-1)

print("Weights after masking (upper triangle = 0):")
for t in range(T):
    row = '  '.join(f'{weights[t,s].item():.2f}' for s in range(T))
    print(f"  pos {t}: [{row}]")
print()
print("Each position only attends to itself and earlier positions.")


### Section 5 — The SingleHeadAttention module


In [ ]:
# ── SingleHeadAttention: the complete module ──────────────────────────────────
#
# This is a clean, self-contained implementation of single-head causal
# self-attention. Every operation matches the equations in the chapter.
#
# Parameters:
#   d_model: embedding dimension of the input tokens
#   d_k    : query and key dimension (head dimension)
#   T_max  : maximum sequence length (for pre-registering the causal mask)
#
# The three learned projections (W_Q, W_K, W_V) are all Linear(d_model, d_k).
# No bias in Q/K/V projections (common practice; bias does not help attention).

class SingleHeadAttention(nn.Module):
    def __init__(self, d_model, d_k, T_max=256):
        super().__init__()
        self.d_k = d_k
        # Learned projections: token embedding → query / key / value space
        self.W_Q = nn.Linear(d_model, d_k, bias=False)
        self.W_K = nn.Linear(d_model, d_k, bias=False)
        self.W_V = nn.Linear(d_model, d_k, bias=False)
        # Register causal mask as a buffer — not a parameter, but saved with the model.
        # tril(ones(T,T)): lower-triangular matrix of True values.
        self.register_buffer('mask', torch.tril(torch.ones(T_max, T_max, dtype=torch.bool)))

    def forward(self, x):
        # x: (batch, T, d_model)
        B, T, _ = x.shape

        Q = self.W_Q(x)   # (B, T, d_k)
        K = self.W_K(x)   # (B, T, d_k)
        V = self.W_V(x)   # (B, T, d_k)

        # Scaled dot-product scores.
        # Q @ K.transpose(-2,-1): for each batch, compute (T,d_k)@(d_k,T) = (T,T).
        scores = Q @ K.transpose(-2, -1) / math.sqrt(self.d_k)   # (B, T, T)

        # Apply causal mask: positions that should not be attended to get -inf.
        # self.mask[:T, :T] slices the pre-registered mask to the current T.
        scores = scores.masked_fill(~self.mask[:T, :T], float('-inf'))

        # Softmax over the key dimension (last axis).
        weights = F.softmax(scores, dim=-1)   # (B, T, T)

        # Weighted aggregation of value vectors.
        out = weights @ V   # (B, T, d_k)
        return out, weights


# ── demo: run one forward pass ─────────────────────────────────────────────
torch.manual_seed(42)
D_MODEL, D_K, T, BATCH = 64, 32, 8, 2

attn = SingleHeadAttention(d_model=D_MODEL, d_k=D_K)
x    = torch.randn(BATCH, T, D_MODEL)
out, w = attn(x)

print(f"Input  shape: {list(x.shape)}   (batch, T, d_model)")
print(f"Output shape: {list(out.shape)}  (batch, T, d_k)")
print(f"Weights shape: {list(w.shape)} (batch, T, T)")
print()
print(f"W_Q params: {attn.W_Q.weight.numel():,}  ({D_MODEL}×{D_K})")
print(f"W_K params: {attn.W_K.weight.numel():,}  ({D_MODEL}×{D_K})")
print(f"W_V params: {attn.W_V.weight.numel():,}  ({D_MODEL}×{D_K})")
print(f"Total attention params: {sum(p.numel() for p in attn.parameters()):,}")
print()
# Verify causal masking: upper triangle of weights should be exactly 0
upper_mass = w[0].triu(diagonal=1).sum().item()
print(f"Weight mass above diagonal: {upper_mass:.6f}  (should be 0.0 — causal mask working)")


---

## Part 3 — Training an Attention Language Model


### Section 6 — The AttentionLM


In [ ]:
# ── AttentionLM: a language model built on single-head self-attention ─────────
#
# Architecture:
#   1. Token embedding   : (vocab_size, d_model)
#   2. Positional embedding: (T_max, d_model)  — added to token embeddings
#   3. SingleHeadAttention: queries/keys/values over the full sequence
#   4. Linear head        : (d_k → vocab_size) — project to vocabulary logits
#
# The positional embedding is necessary because self-attention is order-agnostic:
# the dot product q_t · k_s is the same regardless of whether s precedes or
# follows t. Positional embeddings inject position information so the model
# can distinguish "h at position 3" from "h at position 7".
#
# This is a minimal one-block model — no feedforward sublayer, no layer norm,
# no residual connections. Those arrive in Chapters 9 and 10.

BLOCK_SIZE = 32    # maximum context length (T_max for this model)
D_MODEL    = 64    # token embedding dimension
D_K        = 32    # attention head dimension

class AttentionLM(nn.Module):
    def __init__(self, vocab_size, d_model=D_MODEL, d_k=D_K, block_size=BLOCK_SIZE):
        super().__init__()
        self.block_size   = block_size
        self.token_emb    = nn.Embedding(vocab_size, d_model)
        self.pos_emb      = nn.Embedding(block_size, d_model)
        self.attention    = SingleHeadAttention(d_model, d_k, T_max=block_size)
        self.head         = nn.Linear(d_k, vocab_size)

    def forward(self, x):
        # x: (batch, T) — integer token IDs, T <= block_size
        B, T = x.shape
        # Token + positional embeddings: both (B, T, d_model), summed.
        pos  = torch.arange(T, device=x.device)          # (T,)
        emb  = self.token_emb(x) + self.pos_emb(pos)     # (B, T, d_model)
        # Self-attention: produce context-aware representations.
        ctx, _ = self.attention(emb)                      # (B, T, d_k)
        # Project to vocabulary logits.
        logits = self.head(ctx)                           # (B, T, vocab_size)
        return logits

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0):
        'Autoregressively sample max_new_tokens new tokens.'
        self.eval()
        for _ in range(max_new_tokens):
            idx_ctx = idx[:, -self.block_size:]           # crop to block_size
            logits  = self(idx_ctx)[:, -1, :]             # logits at last position
            probs   = F.softmax(logits / temperature, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1)
            idx     = torch.cat([idx, next_id], dim=1)
        return idx


# ── inspect parameter count ────────────────────────────────────────────────
model = AttentionLM(vocab_size).to(device)
total_params = sum(p.numel() for p in model.parameters())

print("AttentionLM parameter breakdown:")
print(f"  token_emb  : {model.token_emb.weight.numel():>8,}  ({vocab_size} × {D_MODEL})")
print(f"  pos_emb    : {model.pos_emb.weight.numel():>8,}  ({BLOCK_SIZE} × {D_MODEL})")
print(f"  W_Q        : {model.attention.W_Q.weight.numel():>8,}  ({D_MODEL} × {D_K})")
print(f"  W_K        : {model.attention.W_K.weight.numel():>8,}  ({D_MODEL} × {D_K})")
print(f"  W_V        : {model.attention.W_V.weight.numel():>8,}  ({D_MODEL} × {D_K})")
print(f"  head       : {model.head.weight.numel() + model.head.bias.numel():>8,}  ({D_K} × {vocab_size} + bias)")
print(f"  ─────────────────────────")
print(f"  Total      : {total_params:>8,}")
print()
print(f"Context window : {BLOCK_SIZE} tokens")
print(f"Compare Ch 6 k=20 total params: 87,425")


### Section 7 — Training and comparison


In [ ]:
# ── training the AttentionLM ──────────────────────────────────────────────────
#
# We use the same hyperparameters as Chapter 6 where possible, but attention
# requires sequential (not fixed-length) batches — get_batch samples T-length
# windows and produces both inputs and targets as (batch, T) tensors.

N_STEPS    = 5000
BATCH_SIZE = 64
LR         = 1e-3

def get_batch_seq(data, block_size, batch_size):
    'Sample batch_size random windows of length block_size from data.'
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x  = torch.stack([data[i   : i + block_size] for i in ix])
    y  = torch.stack([data[i+1 : i + block_size + 1] for i in ix])
    return x.to(device), y.to(device)

def eval_loss_seq(model, data, block_size, n_samples=512):
    model.eval()
    with torch.no_grad():
        x, y = get_batch_seq(data, block_size, n_samples)
        logits = model(x)   # (B, T, vocab_size)
        # Reshape for cross_entropy: (B*T, vocab_size) and (B*T,)
        loss = F.cross_entropy(logits.view(-1, vocab_size), y.view(-1))
    return loss.item()

# ── train ─────────────────────────────────────────────────────────────────────
torch.manual_seed(42)
model     = AttentionLM(vocab_size).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)

print(f"Training AttentionLM for {N_STEPS} steps...")
loss_curve = []
t0 = time.time()

for step in range(N_STEPS):
    model.train()
    x, y    = get_batch_seq(train_data, BLOCK_SIZE, BATCH_SIZE)
    logits  = model(x)
    loss    = F.cross_entropy(logits.view(-1, vocab_size), y.view(-1))
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if (step + 1) % 500 == 0:
        vl = eval_loss_seq(model, val_data, BLOCK_SIZE)
        loss_curve.append((step + 1, vl))
        print(f"  step {step+1:5d}  val_loss={vl:.4f}  ppl={math.exp(vl):.2f}")

elapsed  = time.time() - t0
val_loss = eval_loss_seq(model, val_data, BLOCK_SIZE, n_samples=2048)
ppl      = math.exp(val_loss)

print()
print(f"Training complete in {elapsed:.1f}s")
print(f"Final val_loss : {val_loss:.4f}")
print(f"Final perplexity: {ppl:.2f}")
print()
print("Chapter 6 reference perplexities (same corpus, 3000 steps):")
print("  k=6  (best fixed-window) : ~9.34")
print("  k=20 (longest context)   : ~11.24")
print(f"  AttentionLM T={BLOCK_SIZE}          : {ppl:.2f}")


### Section 8 — Generated text


In [ ]:
# ── generate text from the trained AttentionLM ───────────────────────────────
SEED      = "First Citizen:\n"
N_CHARS   = 200
seed_ids  = torch.tensor([encode(SEED)], dtype=torch.long).to(device)

model.eval()
out_ids = model.generate(seed_ids, max_new_tokens=N_CHARS, temperature=1.0)
generated = decode(out_ids[0].tolist())

print("Generated text (temperature=1.0):")
print("─" * 60)
print(generated)
print("─" * 60)
print()

# Chapter 6 comparison samples (from notebook ch06)
print("Chapter 6 samples for comparison:")
print("  k=1 : '\nARDYOLI s\nThand nd f nger:\nA n.\nASofatnd...'")
print("  k=6 : (best perplexity, ~9.34)")
print("  k=20: '\nSentle the mords of the sould,\nAnd the...'")


### Section 9 — Inspecting attention patterns


In [ ]:
# ── inspect attention weights on a sample string ─────────────────────────────
#
# What does the model actually attend to? We pass a short string through
# the model and read out the attention weight matrix.
#
# Each row t shows how much position t attended to each position s <= t
# after training. High weight at (t, s) means position t found position s
# relevant for its prediction.

sample_str = "First Citizen"
sample_ids = torch.tensor([encode(sample_str)], dtype=torch.long).to(device)
T_s = len(sample_str)

model.eval()
with torch.no_grad():
    pos  = torch.arange(T_s, device=device)
    emb  = model.token_emb(sample_ids) + model.pos_emb(pos)
    _, attn_weights = model.attention(emb)   # (1, T_s, T_s)

w = attn_weights[0].cpu()   # (T_s, T_s)

print(f"Attention weights for: {repr(sample_str)}")
print()
header = '   ' + ''.join(f'{c:>5}' for c in sample_str)
print(header)
for t, ch_t in enumerate(sample_str):
    row = f"{ch_t!r:3}" + ''.join(
        f'{w[t,s].item():>5.2f}' if s <= t else '   ·  '
        for s in range(T_s)
    )
    print(row)
print()
print("Each row sums to 1.0. Dots (·) mark causally masked positions.")
print("Compare: in FixedWindowModel every positional block was fixed —")
print("these weights are computed fresh from the content of this specific string.")


### Section 10 — What single-head attention cannot do


In [ ]:
# ── one head = one relational subspace ───────────────────────────────────────
#
# A single attention head projects all tokens into ONE query-key space.
# It can learn one pattern of relationships between positions:
#   e.g. "attend to the last vowel" or "attend to the previous punctuation"
#
# But natural language has MULTIPLE simultaneous relationship types:
#   - syntactic dependencies ("subject ↔ verb")
#   - local spelling patterns ("h follows t")
#   - long-range co-references ("the ... it")
#
# A single head must choose — it cannot represent all relationship types
# simultaneously through one set of Q/K projections.
#
# Chapter 8 addresses this with MULTI-HEAD attention:
# H independent heads, each learning a different relational subspace,
# then concatenating their outputs before projecting.

# Visualise: this model has ONE attention head with d_k=32 dimensions.
# It must compress all relevant relationships into this one space.
print("AttentionLM attention head summary:")
print(f"  Number of heads : 1")
print(f"  d_model         : {D_MODEL}")
print(f"  d_k per head    : {D_K}")
print(f"  W_Q, W_K, W_V   : {D_MODEL}×{D_K} each")
print()
print("One head can only learn one relational subspace at a time.")
print("Multi-head attention (Chapter 8) runs H heads in parallel,")
print("each learning a different pattern of cross-position relationships.")
print()
print("Other limitations of this minimal model (deferred to later chapters):")
print("  - No feedforward sublayer         (Chapter 9)")
print("  - No residual connections         (Chapter 10)")
print("  - No layer normalisation          (Chapter 10)")
print("  - Single block (no depth)         (Chapter 11)")


## When does self-attention help?

| Mechanism | What it provides | Limitation |
|---|---|---|
| Fixed-window (Ch 6) | Fixed learned transformation per positional slot | Cannot adapt based on other positions' content |
| Single-head attention (Ch 7) | Content-dependent weighting across all positions | One relational subspace — one pattern at a time |
| Multi-head attention (Ch 8) | H parallel relational subspaces | Concatenation increases cost; still one block |
| Full transformer block (Ch 11) | Attention + FFN + residual + norm | Multiple blocks needed for deep representations |

The key improvement from Ch 6 → Ch 7 is not just better perplexity. It is that the
model can now route information across positions based on content — the same token in
different contexts can attract different attention patterns.

> **Chapter 7 limitation.** A single attention head uses one set of Q/K/V projections.
> It can learn one type of relational pattern at a time. Chapter 8 introduces
> multi-head attention: H independent heads each specialising in a different
> relationship subspace, whose outputs are concatenated before projection.


---

## Lab Exercises


### Exercise 1 — Effect of temperature on attention patterns

Regenerate text at temperature=0.5, 1.0, and 2.0. Compare the outputs.
Does lower temperature produce more repetitive text? Why?


In [ ]:
# ── Exercise 1: temperature effect ───────────────────────────────────────────
seed_ids = torch.tensor([encode("First Citizen:\n")], dtype=torch.long).to(device)

for temp in [0.5, 1.0, 2.0]:
    out = model.generate(seed_ids, max_new_tokens=100, temperature=temp)
    text = decode(out[0].tolist())
    print(f"Temperature {temp}:")
    print(text)
    print()


### Exercise 2 — Attention on different input strings

Inspect the attention weight matrix for three different short strings.
Do the patterns differ? What does the model seem to be attending to?


In [ ]:
# ── Exercise 2: attention patterns for different inputs ───────────────────────
test_strings = ["To be or not", "First Citizen", "KING RICHARD"]

for s in test_strings:
    ids = torch.tensor([encode(s)], dtype=torch.long).to(device)
    T_s = len(s)
    with torch.no_grad():
        pos = torch.arange(T_s, device=device)
        emb = model.token_emb(ids) + model.pos_emb(pos)
        _, w = model.attention(emb)
    w = w[0].cpu()
    print(f"String: {repr(s)}")
    # Print which position each token attends to most (argmax)
    for t, ch in enumerate(s):
        argmax_s = w[t, :t+1].argmax().item()
        print(f"  pos {t} ({ch!r}) → attends most to pos {argmax_s} ({s[argmax_s]!r})  weight={w[t,argmax_s]:.3f}")
    print()


### Exercise 3 — What happens without positional embeddings?

Remove the positional embedding from `AttentionLM.forward` (replace it with zeros).
Train a new model. Does perplexity change? What does this tell you about why
positional embeddings are needed?


In [ ]:
# ── Exercise 3: model without positional embeddings ───────────────────────────
class AttentionLM_NoPE(nn.Module):
    'AttentionLM with positional embeddings zeroed out.'
    def __init__(self, vocab_size, d_model=D_MODEL, d_k=D_K, block_size=BLOCK_SIZE):
        super().__init__()
        self.block_size = block_size
        self.token_emb  = nn.Embedding(vocab_size, d_model)
        self.attention  = SingleHeadAttention(d_model, d_k, T_max=block_size)
        self.head       = nn.Linear(d_k, vocab_size)

    def forward(self, x):
        B, T = x.shape
        emb  = self.token_emb(x)           # NO positional embedding
        ctx, _ = self.attention(emb)
        return self.head(ctx)

torch.manual_seed(42)
model_nope = AttentionLM_NoPE(vocab_size).to(device)
opt_nope   = torch.optim.AdamW(model_nope.parameters(), lr=LR)

for step in range(N_STEPS):
    model_nope.train()
    x, y   = get_batch_seq(train_data, BLOCK_SIZE, BATCH_SIZE)
    logits = model_nope(x)
    loss   = F.cross_entropy(logits.view(-1, vocab_size), y.view(-1))
    opt_nope.zero_grad(); loss.backward(); opt_nope.step()

vl_nope = eval_loss_seq(model_nope, val_data, BLOCK_SIZE, n_samples=2048)
vl_pe   = eval_loss_seq(model,      val_data, BLOCK_SIZE, n_samples=2048)

print(f"With positional embeddings   : val_loss={vl_pe:.4f}  ppl={math.exp(vl_pe):.2f}")
print(f"Without positional embeddings: val_loss={vl_nope:.4f}  ppl={math.exp(vl_nope):.2f}")
print()
print("Observation: without positional embeddings, the model cannot distinguish")
print("'th' from 'ht' — order information is lost.")


### Exercise 4 — What does each training step cost?

Measure wall-clock time per training step for AttentionLM vs FixedWindowModel (k=20).
Why is attention more expensive? How does cost scale with context length T?


In [ ]:
# ── Exercise 4: cost comparison ───────────────────────────────────────────────
import time

def time_steps(model_fn, n=100):
    'Time n forward+backward steps, return avg ms per step.'
    opt = torch.optim.AdamW(model_fn.parameters(), lr=1e-3)
    t0 = time.time()
    for _ in range(n):
        model_fn.train()
        x, y = get_batch_seq(train_data, BLOCK_SIZE, BATCH_SIZE)
        if hasattr(model_fn, 'k'):   # FixedWindowModel expects (B, k) not (B, T)
            x2 = x[:, -model_fn.k:]
            y2 = y[:, -1]
            loss = F.cross_entropy(model_fn(x2), y2)
        else:
            logits = model_fn(x)
            loss   = F.cross_entropy(logits.view(-1, vocab_size), y.view(-1))
        opt.zero_grad(); loss.backward(); opt.step()
    return (time.time() - t0) / n * 1000   # ms per step

fw_k20 = FixedWindowModel(vocab_size, k=20).to(device)
attn   = AttentionLM(vocab_size).to(device)

ms_fw   = time_steps(fw_k20)
ms_attn = time_steps(attn)

print(f"FixedWindowModel k=20 : {ms_fw:.2f} ms/step")
print(f"AttentionLM T={BLOCK_SIZE}     : {ms_attn:.2f} ms/step")
print()
print("Attention cost: O(T^2 * d_k) for the score matrix vs O(k * d) for fixed-window.")
print(f"With T={BLOCK_SIZE}: score matrix is ({BLOCK_SIZE}×{BLOCK_SIZE}) per batch element.")


### Exercise 5 — Perplexity vs context length

Train AttentionLM models with block_size ∈ {4, 8, 16, 32, 64}.
Plot (or print) validation perplexity vs block_size.
Does attention benefit more from longer context than the fixed-window model did?


In [ ]:
# ── Exercise 5: perplexity vs block_size for AttentionLM ──────────────────────
BLOCK_SIZES = [4, 8, 16, 32, 64]
results_attn = {}

torch.manual_seed(42)
for bs in BLOCK_SIZES:
    m   = AttentionLM(vocab_size, block_size=bs).to(device)
    opt = torch.optim.AdamW(m.parameters(), lr=LR)
    for step in range(3000):
        m.train()
        x, y   = get_batch_seq(train_data, bs, BATCH_SIZE)
        logits = m(x)
        loss   = F.cross_entropy(logits.view(-1, vocab_size), y.view(-1))
        opt.zero_grad(); loss.backward(); opt.step()
    vl  = eval_loss_seq(m, val_data, bs, n_samples=2048)
    ppl = math.exp(vl)
    results_attn[bs] = ppl
    print(f"block_size={bs:3d}  val_loss={vl:.4f}  ppl={ppl:.2f}")

print()
print("Ch 6 FixedWindowModel for comparison:")
print("  k=6  best: ~9.34")
print("  k=20     : ~11.24")
print()
print("Does attention degrade at large context the way the fixed-window model did?")


---

## Research Connections

**Vaswani et al. (2017)** — *Attention Is All You Need*

The paper that introduced the Transformer architecture.
The core equation implemented in this chapter — scaled dot-product attention —
appears in Section 3.2.1:

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right) V$$

Vaswani et al. motivate dividing by $\sqrt{d_k}$ exactly as Section 3 of this notebook
demonstrates: without scaling, the dot products grow large in magnitude for large $d_k$,
pushing the softmax into regions with extremely small gradients.

The full Transformer stacks multi-head attention (Chapter 8) with a feedforward sublayer
(Chapter 9), residual connections, and layer normalisation (Chapter 10). This chapter
builds only the attention sub-component.

**Bahdanau et al. (2015)** — *Neural Machine Translation by Jointly Learning to Align and Translate*

The paper that introduced attention for sequence-to-sequence models before the Transformer.
Bahdanau's mechanism computed alignment scores between encoder hidden states and a decoder
query — the predecessor of the query-key formulation implemented here.
The Transformer generalises this to self-attention: every position can attend to every other
position in the same sequence, not just decoder-to-encoder.

**Paper 2 — QCCR (Query-Conditional Context Rank)**

The Chapter 2 research paper asks: given a query token, what is the minimum rank needed to
approximate the KV cache for that query? This connects directly to the attention mechanism
implemented here — the rank of the attention score matrix determines the effective dimensionality
of the information each query retrieves. Lab Exercise 5 provides empirical evidence for how
context length affects predictive performance under attention.


---

## What This Chapter Deliberately Does Not Explain

**Multi-head attention.** This chapter implements a single attention head.
Real transformers use H parallel heads, each learning a different relational subspace.
Chapter 8 adds this.

**Feedforward sublayer.** The Transformer block contains an attention sublayer AND
a position-wise feedforward network. This chapter's `AttentionLM` has no FFN.
Chapter 9 adds this.

**Residual connections and layer normalisation.** Deep networks require both to
train stably. Chapter 10 adds these. Without them, this one-block model trains fine
but a stack of blocks would not.

**Relative positional encodings.** This chapter uses learned absolute positional
embeddings (one learnable vector per position slot). Modern transformers use relative
encodings (RoPE, ALiBi) that generalise to sequence lengths not seen during training.
Chapter 11 discusses alternatives.

**Cross-attention.** The attention mechanism here is self-attention: queries, keys, and
values all come from the same sequence. Encoder-decoder transformers use cross-attention
where queries come from the decoder and keys/values come from the encoder.
Not covered in this book's scope.

**Flash Attention.** The $O(T^2)$ memory cost of materialising the full attention score
matrix becomes prohibitive for long sequences. Flash Attention reorders the computation
to avoid storing the full matrix. Not covered here — the standard formulation is correct
and sufficient for understanding the mechanism.
